In [5]:
import pandas as pd
from getpass import getpass
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

In [8]:
df = pd.read_csv("Sales.csv")
df["order_id"] = df["order_id"].astype(int)
df["customer_id"] = df["customer_id"].astype(int)
df["product_id"] = df["product_id"].astype(int)
df["quantity"] = df["quantity"].astype(int)
df["unit_price"] = df["unit_price"].astype(float)
df["total_amount"] = df["total_amount"].astype(float)
print(df.head(10))
print(df.shape)
print(df.dtypes)

   order_id  customer_id  product_id  quantity  unit_price  total_amount
0      1001            1         101         2     25000.0       50000.0
1      1002            2         102         1       500.0         500.0
2      1004            4         101         1     25000.0       25000.0
3      1005            1         104         2      7000.0       14000.0
4      1007            2         105         4      2500.0       10000.0
5      1009            4         103         2      1200.0        2400.0
6      1010            1         104         1      7000.0        7000.0
(7, 6)
order_id          int64
customer_id       int64
product_id        int64
quantity          int64
unit_price      float64
total_amount    float64
dtype: object


In [ ]:
connection_url = URL.create(
    "mssql+pyodbc",
    username="admn",
    password=getpass(""),
    host="covid19prosrv.database.windows.net",
    port=1433,
    database="lms_r5",
    query={
        "driver": "ODBC Driver 18 for SQL Server",
        "Encrypt": "yes",
        "TrustServerCertificate": "no",
    },
)

In [14]:
engine = create_engine(
    connection_url,
    fast_executemany=True,
)

In [15]:
try:
    with engine.begin() as connection:
        print("Connected:", connection.execute(
            text("SELECT DB_NAME()")
        ).scalar())

        df.to_sql(
            name="Sales",
            schema="dbo",
            con=connection,
            if_exists="append",
            index=False,
            chunksize=1000,
        )

    print(f"Loaded {len(df)} rows.")
finally:
    engine.dispose()

Connected: lms_r5
Loaded 7 rows.
